In [1]:
import json

treatments = {
    "Tomato_Early_blight": {
        "severity": "Moderate",
        "description": "Fungal disease causing dark spots with concentric rings on lower leaves.",
        "treatment": "Apply copper-based fungicide every 7-10 days. Remove infected leaves immediately.",
        "prevention": "Rotate crops yearly. Avoid overhead watering. Ensure good air circulation.",
        "urgency": "Act within 3-5 days"
    },
    "Tomato_Late_blight": {
        "severity": "High",
        "description": "Aggressive fungal disease causing water-soaked lesions on leaves and stems.",
        "treatment": "Apply chlorothalonil or mancozeb fungicide immediately. Remove and destroy infected plants.",
        "prevention": "Use resistant varieties. Avoid wet foliage. Apply preventive fungicide in humid weather.",
        "urgency": "Act within 24 hours"
    },
    "Tomato_Leaf_Mold": {
        "severity": "Moderate",
        "description": "Fungal disease causing yellow patches on upper leaf surface and olive-green mold below.",
        "treatment": "Apply fungicide containing chlorothalonil. Improve greenhouse ventilation.",
        "prevention": "Reduce humidity below 85%. Space plants for airflow. Avoid wetting leaves.",
        "urgency": "Act within 3-5 days"
    },
    "Tomato_Septoria_leaf_spot": {
        "severity": "Moderate",
        "description": "Fungal disease causing small circular spots with dark borders on leaves.",
        "treatment": "Apply copper or chlorothalonil fungicide. Remove infected lower leaves.",
        "prevention": "Mulch around plants. Rotate crops. Stake plants to improve air circulation.",
        "urgency": "Act within 3-5 days"
    },
    "Tomato_Spider_mites_Two_spotted_spider_mite": {
        "severity": "Moderate",
        "description": "Tiny mites causing stippled yellowing on leaves, especially in hot dry conditions.",
        "treatment": "Apply miticide or neem oil spray. Increase humidity around plants.",
        "prevention": "Monitor regularly. Avoid water stress. Introduce predatory mites as biocontrol.",
        "urgency": "Act within 3-5 days"
    },
    "Tomato__Target_Spot": {
        "severity": "Moderate",
        "description": "Fungal disease causing circular target-like lesions on leaves and fruit.",
        "treatment": "Apply azoxystrobin or chlorothalonil fungicide.",
        "prevention": "Remove crop debris. Rotate crops. Avoid overhead irrigation.",
        "urgency": "Act within 3-5 days"
    },
    "Tomato__Tomato_YellowLeaf__Curl_Virus": {
        "severity": "High",
        "description": "Viral disease spread by whiteflies causing leaf curling and yellowing.",
        "treatment": "No cure available. Remove and destroy infected plants immediately to prevent spread.",
        "prevention": "Control whitefly populations. Use reflective mulch. Plant resistant varieties.",
        "urgency": "Act within 24 hours"
    },
    "Tomato__Tomato_mosaic_virus": {
        "severity": "High",
        "description": "Viral disease causing mottled light and dark green patterns on leaves.",
        "treatment": "No cure available. Remove infected plants. Disinfect tools with bleach solution.",
        "prevention": "Use certified disease-free seeds. Wash hands before handling plants.",
        "urgency": "Act within 24 hours"
    },
    "Tomato_Bacterial_spot": {
        "severity": "Moderate",
        "description": "Bacterial disease causing small water-soaked spots on leaves and fruit.",
        "treatment": "Apply copper-based bactericide. Remove heavily infected plant parts.",
        "prevention": "Use disease-free seeds. Avoid overhead irrigation. Rotate crops.",
        "urgency": "Act within 3-5 days"
    },
    "Tomato_healthy": {
        "severity": "None",
        "description": "Plant appears healthy with no signs of disease.",
        "treatment": "No treatment needed. Continue regular care.",
        "prevention": "Maintain regular watering, fertilization and monitoring schedule.",
        "urgency": "No action needed"
    },
    "Potato___Early_blight": {
        "severity": "Moderate",
        "description": "Fungal disease causing dark spots with yellow halos on older leaves.",
        "treatment": "Apply mancozeb or chlorothalonil fungicide every 7-14 days.",
        "prevention": "Use certified seed potatoes. Rotate crops. Maintain plant nutrition.",
        "urgency": "Act within 3-5 days"
    },
    "Potato___Late_blight": {
        "severity": "High",
        "description": "Devastating fungal disease that can destroy entire crop within days.",
        "treatment": "Apply metalaxyl or cymoxanil fungicide immediately. Destroy infected plants.",
        "prevention": "Use resistant varieties. Apply preventive fungicide. Avoid overhead irrigation.",
        "urgency": "Act within 24 hours"
    },
    "Potato___healthy": {
        "severity": "None",
        "description": "Plant appears healthy with no signs of disease.",
        "treatment": "No treatment needed. Continue regular care.",
        "prevention": "Maintain regular watering, fertilization and monitoring schedule.",
        "urgency": "No action needed"
    },
    "Pepper__bell___Bacterial_spot": {
        "severity": "Moderate",
        "description": "Bacterial disease causing water-soaked lesions on leaves and fruit.",
        "treatment": "Apply copper-based bactericide. Remove infected plant material.",
        "prevention": "Use disease-free seeds. Avoid working with wet plants. Rotate crops.",
        "urgency": "Act within 3-5 days"
    },
    "Pepper__bell___healthy": {
        "severity": "None",
        "description": "Plant appears healthy with no signs of disease.",
        "treatment": "No treatment needed. Continue regular care.",
        "prevention": "Maintain regular watering, fertilization and monitoring schedule.",
        "urgency": "No action needed"
    }
}

with open('data/treatments.json', 'w') as f:
    json.dump(treatments, f, indent=2)

print(f"Treatments saved for {len(treatments)} disease classes!")

Treatments saved for 15 disease classes!


In [2]:
api_code = '''import torch
import torch.nn as nn
import numpy as np
import joblib
import json
import pickle
import io
from torchvision import models, transforms
from PIL import Image
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional

app = FastAPI(title="Crop Disease AI", version="1.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Load everything at startup ──
with open("data/processed/data_splits.pkl", "rb") as f:
    image_data = pickle.load(f)
IMAGE_CLASSES = image_data["classes"]

with open("data/treatments.json") as f:
    TREATMENTS = json.load(f)

# Image transform
TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load ResNet50 feature extractor
resnet = models.resnet50(weights=None)
resnet.fc = nn.Linear(2048, len(IMAGE_CLASSES))
resnet.load_state_dict(torch.load("models/resnet50_finetuned.pth",
                                   map_location="cpu"))
EMBEDDING_MODEL = nn.Sequential(*list(resnet.children())[:-1])
EMBEDDING_MODEL.eval()

# Load fusion MLP
class FusionMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.network(x)

FUSION_MODEL = FusionMLP(input_dim=2057, num_classes=15)
FUSION_MODEL.load_state_dict(torch.load("models/fusion_mlp_best.pth",
                                         map_location="cpu"))
FUSION_MODEL.eval()

# Load scaler
SCALER = joblib.load("models/tabular_scaler_v2.pkl")

print("All models loaded successfully!")

# ── Request schema ──
class TabularFeatures(BaseModel):
    soil_pH: float
    nitrogen: float
    phosphorus: float
    potassium: float
    temperature: float
    humidity: float
    rainfall: float
    crop_age_days: float
    sunlight_hours: float

# ── Endpoints ──
@app.get("/")
def root():
    return {"message": "Crop Disease AI API is running!", "version": "1.0"}

@app.get("/classes")
def get_classes():
    return {"classes": IMAGE_CLASSES, "total": len(IMAGE_CLASSES)}

@app.post("/predict")
async def predict(
    file: UploadFile = File(...),
    soil_pH: float = 6.5,
    nitrogen: float = 80.0,
    phosphorus: float = 60.0,
    potassium: float = 70.0,
    temperature: float = 28.0,
    humidity: float = 65.0,
    rainfall: float = 120.0,
    crop_age_days: float = 45.0,
    sunlight_hours: float = 7.0
):
    # Validate image
    if not file.content_type.startswith("image/"):
        raise HTTPException(status_code=400, detail="File must be an image")

    # Load and preprocess image
    contents = await file.read()
    img = Image.open(io.BytesIO(contents)).convert("RGB")
    img_tensor = TRANSFORM(img).unsqueeze(0)

    # Get image embedding
    with torch.no_grad():
        embedding = EMBEDDING_MODEL(img_tensor)
        embedding = embedding.squeeze(-1).squeeze(-1).numpy()

    # Preprocess tabular
    tab_values = np.array([[soil_pH, nitrogen, phosphorus, potassium,
                            temperature, humidity, rainfall,
                            crop_age_days, sunlight_hours]])
    tab_scaled = SCALER.transform(tab_values)

    # Fuse and predict
    fused = np.concatenate([embedding, tab_scaled], axis=1)
    fused_tensor = torch.tensor(fused, dtype=torch.float32)

    with torch.no_grad():
        outputs = FUSION_MODEL(fused_tensor)
        probs = torch.softmax(outputs, dim=1)
        confidence, predicted = probs.max(1)

    disease = IMAGE_CLASSES[predicted.item()]
    confidence_pct = round(confidence.item() * 100, 2)

    # Get treatment
    treatment = TREATMENTS.get(disease, {
        "severity": "Unknown",
        "description": "No information available.",
        "treatment": "Consult an agricultural expert.",
        "prevention": "Monitor crop regularly.",
        "urgency": "Consult expert"
    })

    return {
        "disease": disease,
        "confidence": confidence_pct,
        "severity": treatment["severity"],
        "description": treatment["description"],
        "treatment": treatment["treatment"],
        "prevention": treatment["prevention"],
        "urgency": treatment["urgency"]
    }

@app.get("/health")
def health():
    return {"status": "healthy"}
'''

with open("src/api.py", "w") as f:
    f.write(api_code)

print("src/api.py created!")

src/api.py created!


In [3]:
new_api = open('src/api.py').read().replace(
    '    # Load and preprocess image\n    contents = await file.read()\n    img = Image.open(io.BytesIO(contents)).convert("RGB")',
    '''    # Load and preprocess image
    contents = await file.read()
    import tempfile, os
    suffix = os.path.splitext(file.filename)[-1] or ".jpg"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(contents)
        tmp_path = tmp.name
    try:
        img = Image.open(tmp_path).convert("RGB")
    except Exception:
        raise HTTPException(status_code=400, detail="Could not read image. Please upload a JPG or PNG file.")
    finally:
        os.unlink(tmp_path)'''
)

with open('src/api.py', 'w') as f:
    f.write(new_api)

print("src/api.py updated!")

src/api.py updated!
